# HIGGS - exploration and interaction information matrix

This notebook generates a sample from the **HIGGS** dataset (Baldi et al., 2014; UCI 280)
using `generate_higgs_data`, performs basic exploration (shape, class balance, missing values,
distributions, and correlations), and computes the **interaction information matrix**
`II(Xi; Xj; y)` using `compute_information_matrices`.

HIGGS features are continuous, so they are discretized when computing information
measures (`n_bins`).

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

import dotenv
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import PowerTransformer

from my_project.data.load_higgs import generate_higgs_data
from my_project.experiments.synthetic import higgs_pipeline
from my_project.information_theory import compute_information_matrices
from my_project.reporting import (
    pair_table_from_matrix,
    pairwise_ii_mi_table,
    plot_category_barplots,
    plot_class_balance,
    plot_feature_histograms,
    plot_ii_by_edges,
    plot_ii_graph,
    save_figure,
    save_table,
)

RANDOM_STATE = 42

# Shared plot style: use the same theme and palette as in the other notebooks.
# Show only horizontal grid lines (vertical lines are not useful for categorical axes).
sns.set_theme(style="whitegrid", context="notebook", rc={"axes.grid.axis": "y"})
PALETTE = "husl"  # CATEGORICAL variables - shared palette
PALETTE_SEQ = "crest"  # ORDERED variables - sequential palette

In [ ]:
# LaTeX export uses shared functions from my_project.reporting.latex_export.
# figures -> $RESULTS_LATEX_DIR/plots/*.pdf, tables -> $RESULTS_LATEX_DIR/tables/*.tex
dotenv.load_dotenv(override=True)
RESULTS_LATEX_DIR = os.getenv("RESULTS_LATEX_DIR")

POLISH_COLUMN_NAMES = {
    "feature": "zmienna",
    "feature_i": "zmienna i",
    "feature_j": "zmienna j",
    "mean": "średnia",
    "median": "mediana",
    "std": "odch. stand.",
    "min": "min.",
    "max": "maks.",
    "skewness": "wsp. skośności",
    "mutual_information": "informacja wzajemna",
    "interaction_information": "informacja interakcyjna",
}

## Data generation

We draw a small sample (`n_samples`) from the leading-row pool
(using `random_state`). The default is `feature_set="low_level"` (21 low-level
kinematic features); it can be changed to `"high_level"` or `"all"`.

In [ ]:
data = generate_higgs_data(
    n_samples=20_000,
    random_state=RANDOM_STATE,
    feature_set="low_level",
)

X = data["X"]
y = data["y"]

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
X.head()

## Basic exploration

In [ ]:
len(X.columns)

In [ ]:
X.columns

In [ ]:
columns_order = [
    "missing_energy_magnitude",
    "lepton_pt",
    "jet1_pt",
    "jet2_pt",
    "jet3_pt",
    "jet4_pt",
    "missing_energy_phi",
    "lepton_phi",
    "jet1_phi",
    "jet2_phi",
    "jet3_phi",
    "jet4_phi",
    "lepton_eta",
    "jet1_eta",
    "jet2_eta",
    "jet3_eta",
    "jet4_eta",
    "jet1_btag",
    "jet2_btag",
    "jet3_btag",
    "jet4_btag",
]

In [ ]:
# Descriptive feature statistics.
X.describe().T

In [ ]:
# Missing values and column data types.
pd.DataFrame(
    {
        # "dtype": X.dtypes.astype(str),
        # "missing_pct": X.isna().mean(),
        "n_unique": X.nunique(),
        "min": X.min(),
        "mean": X.mean(),
        "max": X.max(),
        "skewness": X.skew(),
    }
)

In [ ]:
# Descriptive feature statistics: mean, median, missing count, and unique-value count.
descriptive_stats = pd.DataFrame(
    {
        "mean": X.mean(),
        "median": X.median(),
        "std": X.std(),
        "min": X.min(),
        "max": X.max(),
        "skewness": X.skew(),
        # "n_missing": X.isna().sum(),
        # "n_unique": X.nunique(),
    }
)
descriptive_stats.index.name = "feature"
descriptive_stats = descriptive_stats.sort_index()
save_table(
    descriptive_stats.round(3).rename(columns=POLISH_COLUMN_NAMES).reset_index(),
    "higgs_eda_descriptive_stats",
    index=False,
)
descriptive_stats.round(3).rename(columns=POLISH_COLUMN_NAMES).reset_index()

In [ ]:
y.mean()

In [ ]:
# Class balance: HIGGS is a binary problem (1 = signal, 0 = background).
class_counts = y.value_counts().sort_index()
print(class_counts)
print(f"\nudział klasy 1: {y.mean():.3f}")

plot_class_balance(y, title="Balans klas HIGGS")
plt.tight_layout()
plt.show()

In [ ]:
# Pairplot for features with the strongest interaction with the target. A full pairplot
# of all 21 features would be unreadable, so select features found in pairs with the
# highest interaction information II(Xi; Xj; y), using a row sample for speed.
N_PAIR_FEATURES = 5

ii_rank = compute_information_matrices(
    X,
    y,
    measures="interaction_information",
    n_bins=8,
    n_jobs=-1,
)["interaction_information"]
top_pairs = pair_table_from_matrix(ii_rank)

pair_features = []
for fi, fj in zip(top_pairs["feature_i"], top_pairs["feature_j"]):
    for feature in (fi, fj):
        if feature not in pair_features:
            pair_features.append(feature)
    if len(pair_features) >= N_PAIR_FEATURES:
        break
pair_features = pair_features[:N_PAIR_FEATURES]
print("Cechy wybrane wg najwyższej II:", pair_features)

plot_df = X[pair_features].sample(n=2000, random_state=RANDOM_STATE).copy()
plot_df["y"] = y.loc[plot_df.index].astype(str)

grid = sns.pairplot(
    plot_df,
    kind="scatter",
    hue="y",
    corner=True,
    diag_kind="hist",
    plot_kws={"s": 10, "alpha": 0.3, "edgecolor": "none"},
    diag_kws={"bins": 40, "element": "step", "common_norm": False},
)
# Place the legend inside the empty upper-right corner left by corner=True.
sns.move_legend(grid, "upper right", bbox_to_anchor=(0.9, 0.9), frameon=True, title="y")
grid.figure.suptitle("Pairplot wybranych cech wg klasy", y=1.02)
plt.show()

In [ ]:
# Histograms of all features arranged in a grid (colored by class).
HISTOGRAM_FONT_SIZE = 25
HISTOGRAM_TITLE_PAD = 14
HISTOGRAM_SUBPLOT_SIZE = (5, 5)
HISTOGRAM_WSPACE = 0.4
HISTOGRAM_HSPACE = 0.5
fig = plot_feature_histograms(
    X=X[columns_order],
    y=None,
    legend_bbox_y=-0.07,
    by_columns=True,
    n_cols=7,
    subplot_size=HISTOGRAM_SUBPLOT_SIZE,
    wspace=HISTOGRAM_WSPACE,
    hspace=HISTOGRAM_HSPACE,
    font_size=HISTOGRAM_FONT_SIZE,
    title_pad=HISTOGRAM_TITLE_PAD,
)
# for ax in fig.axes:
#     for patch in ax.patches:
#         patch.set_edgecolor("black")
save_figure(fig, "higgs_eda_feature_histograms")
plt.show()
len(columns_order)

## Data after preprocessing

The same features after passing through `higgs_pipeline` (from `experiments/synthetic`).
The histograms show transformed continuous features (Yeo-Johnson / standardization),
while the bar plots show category counts for the discrete `*_btag` flags.


In [ ]:
# higgs_pipeline (as in experiments/synthetic): Yeo-Johnson for skewed features,
# standardization for the other continuous features, and *_btag flags as ordered categories.
X_preprocessed = higgs_pipeline.fit_transform(X, y)

# Preserve the feature order from columns_order (discrete variables last).
num_set = set(X_preprocessed.select_dtypes(include="number").columns)
cat_set = set(X_preprocessed.select_dtypes(exclude="number").columns)
num_cols = [c for c in columns_order if c in num_set]
categorical_cols = [c for c in columns_order if c in cat_set]
X_preprocessed.dtypes

In [ ]:
# Histograms of continuous features after transformation (higgs_pipeline).
fig = plot_feature_histograms(
    X=X_preprocessed,
    y=None,
    columns=num_cols,
    legend_bbox_y=-0.07,
    by_columns=True,
    n_cols=6,
)
save_figure(fig, "higgs_eda_feature_histograms_transformed")
plt.show()

In [ ]:
# Category-count bar plots for discrete features (*_btag flags).
fig = plot_category_barplots(
    X_preprocessed,
    y=None,
    columns=categorical_cols,
    n_cols=2,
    category_order=["1.0", "2.0", "3.0"],
)
save_figure(fig, "higgs_eda_category_barplots")
plt.show()

In [ ]:
X["lepton_pt"].hist(bins=50, color="C0", alpha=0.7)

In [ ]:
pt = PowerTransformer(method="box-cox", standardize=True, copy=True)

In [ ]:
pd.DataFrame(pt.fit_transform(X[["jet4_pt"]]), columns=["jet4_pt"]).hist(
    bins=50, color="steelblue", edgecolor="black"
)

## Interaction information matrix

`compute_information_matrices` computes information-theoretic measures for every
feature pair. We are primarily interested in **interaction information**
`II(Xi; Xj; y)` (pyitlib convention: a positive value indicates pairwise **synergy**
with respect to `y`). For context, we also compute `joint_target_mi`,
`I((Xi, Xj); y)`, and `feature_mi`, `I(Xi; Xj)`.

The features are continuous, so they are discretized (`n_bins`).

In [ ]:
matrices = compute_information_matrices(
    X,
    y,
    measures=("feature_mi", "joint_target_mi", "interaction_information"),
    n_bins=8,
    n_jobs=-1,
)
matrices.keys()

In [ ]:
figsize = (11, 9)

scale = 0.75
figsize = (figsize[0] * scale, figsize[1] * scale)

In [ ]:
# Interaction information matrix II(Xi; Xj; y).
ii = matrices["interaction_information"]

fig, ax = plt.subplots(figsize=figsize)
sns.heatmap(
    ii,
    cmap="viridis",
    # center=0,
    square=True,
    ax=ax,
)
# ax.set_title("Informacja interakcyjna par zmiennych objaśniających i zmiennej celu")
fig.tight_layout()
save_figure(fig, "higgs_eda_interaction_information")
plt.show()

In [ ]:
name = "feature_mi"
fig, ax = plt.subplots(figsize=figsize)
sns.heatmap(matrices[name], cmap="viridis", square=True, ax=ax)
# ax.set_title("Informacja wzajemna między zmiennymi objaśniającymi")
fig.tight_layout()
save_figure(fig, "higgs_eda_mutual_information")

plt.show()

In [ ]:
# Feature-pair table with II(Xi; Xj; y) and I(Xi; Xj), sorted by decreasing II.
pair_table = pairwise_ii_mi_table(ii, matrices["feature_mi"])
pair_table.head(15)

In [ ]:
pair_table.index = pair_table.index + 1

save_table(
    pair_table.head(10)
    .drop(columns=["mutual_information"])
    .round(4)
    .rename(columns=POLISH_COLUMN_NAMES),
    "higgs_eda_pairwise_ii",
    # index=False
)

In [ ]:
save_table(
    pair_table.head(10).round(4).rename(columns=POLISH_COLUMN_NAMES),
    "higgs_eda_pairwise_ii_mi",
    # index=False
)
pair_table

In [ ]:
# Interaction information for feature-pair edges, sorted in decreasing order.
fig, ax = plt.subplots(figsize=(9, 5))
plot_ii_by_edges(
    pair_table,
    ax=ax,
    xlabel="Pozycja w rankingu\n(%)",
    ylabel="Informacja interakcyjna (II)",
)
fig.tight_layout()
save_figure(fig, "higgs_eda_interaction_information_by_edges")
plt.show()

In [ ]:
# Graph of feature pairs with the highest II.
top_pct = 0.3
n_pairs = pair_table.shape[0]
k = int(n_pairs * top_pct)
actual_pct = k / n_pairs
pct_prefix = "" if np.isclose(actual_pct, top_pct) else "około "

fig, ax = plt.subplots(figsize=(11, 9))
graph = plot_ii_graph(
    pair_table,
    ii.columns,
    k=k,
    ax=ax,
    seed=RANDOM_STATE,
    layout="circular",
    curved_edges=False,
    label_fontsize=12,
    node_size=400,
)
print(
    f"k={k}: {graph.number_of_nodes()} węzłów, {graph.number_of_edges()} krawędzi "
    f"(z {n_pairs} możliwych); "
    f"próg II = {pair_table['interaction_information'].iloc[k - 1]:.4f}"
)
fig.tight_layout()
ax.set_title(
    f"top {k} ({pct_prefix}{top_pct * 100:.0f}%) krawędzi",
    y=0.97,
    fontsize=14,
)
save_figure(fig, "higgs_eda_interaction_graph")
plt.show()